# 01 — Bronze | MLB Stats API Extraction (incremental + backfill)

**Project:** MLB Performance Intelligence Platform

Downloads every endpoint the platform needs and lands each response, unchanged, as a JSON
envelope in the append-only Volume `bronze.mlb_api_raw`. The next notebook ingests those files
into Bronze with Auto Loader.

| Endpoint | What it gives | When it is requested |
|---|---|---|
| `/schedule` | Games, status, final score, line score (R/H/E/LOB), winning/losing/save pitcher | Every date in the window |
| `/game/{gamePk}/boxscore` | Every player's batting and pitching line, attendance, duration, umpires | Each completed game not yet extracted (daily mode also re-checks the lookback window for scoring corrections) |
| `/people` | Player profile: birth country, position, bats/throws, debut | Players seen in new boxscores and not yet in `silver.dim_player` |
| `/teams` | League, division, venue, abbreviations | Each season in the window |
| `/venues` | Ballpark city, country, coordinates, elevation | New venues |
| `/standings` | Official W-L and runs per team (used to reconcile our numbers) | Each season in the window |

**Modes** (widgets or Job parameters)
- `daily` (default): yesterday plus `lookback_days` (America/Santo_Domingo), so late scoring changes and suspended games are picked up.
- `backfill`: full seasons listed in `seasons` (default 2024–2026), in monthly schedule chunks. Re-running is safe: games already extracted are skipped.
- `start_date` / `end_date`: custom range, overrides both.

**Idempotency:** each response is hashed (SHA-256 of canonical JSON). If it equals the last extraction of the same request it is registered as `UNCHANGED` and not landed again.

In [ ]:
dbutils.widgets.dropdown("mode", "daily", ["daily", "backfill"], "Mode")
dbutils.widgets.text("seasons", "2024,2025,2026", "Backfill seasons")
dbutils.widgets.text("lookback_days", "3", "Daily lookback (days)")
dbutils.widgets.text("start_date", "", "Custom start YYYY-MM-DD (optional)")
dbutils.widgets.text("end_date", "", "Custom end YYYY-MM-DD (optional)")
dbutils.widgets.text("max_workers", "8", "Concurrent requests")
dbutils.widgets.dropdown("refresh_people", "false", ["false", "true"], "Re-download every player profile")

In [ ]:
import sys, importlib
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
import mlb_api as API
for _m in (C, P, API):
    importlib.reload(_m)

from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo

MODE = dbutils.widgets.get("mode")
SEASONS = [int(s) for s in dbutils.widgets.get("seasons").split(",") if s.strip()]
LOOKBACK = int(dbutils.widgets.get("lookback_days"))
START, END = dbutils.widgets.get("start_date").strip(), dbutils.widgets.get("end_date").strip()
MAX_WORKERS = int(dbutils.widgets.get("max_workers"))
REFRESH_PEOPLE = dbutils.widgets.get("refresh_people") == "true"
RUN_ID = P.new_run_id()

yesterday = datetime.now(ZoneInfo(C.TIMEZONE)).date() - timedelta(days=1)
if START:
    windows = API.date_chunks(date.fromisoformat(START), date.fromisoformat(END or START))
elif MODE == "backfill":
    windows = [c for s in SEASONS if (w := API.season_window(s, yesterday)) for c in API.date_chunks(*w)]
else:
    windows = [(yesterday - timedelta(days=LOOKBACK), yesterday)]
WINDOW_LABEL = f"{windows[0][0]}..{windows[-1][1]}" if windows else "empty"
print(f"Run {RUN_ID} | mode={'custom' if START else MODE} | {len(windows)} schedule request(s) covering {WINDOW_LABEL}")

## Extraction engine
`fetch()` runs a list of calls concurrently, lands new content and records every request in
`metadata.extract_register`. Payloads are handed to a callback and then dropped, so a full-season
backfill (~2,500 boxscores per season) never holds all responses in driver memory.

In [ ]:
client = API.MlbApiClient()

# Last successful SHA-256 per request, to skip unchanged content
_last = spark.sql(f"""
    SELECT endpoint, request_key, payload_sha256 FROM (
      SELECT *, ROW_NUMBER() OVER (PARTITION BY endpoint, request_key ORDER BY extracted_at DESC) rn
      FROM {C.EXTRACT_REGISTER} WHERE status_code = 200) WHERE rn = 1""").collect()
last_sha = {(r.endpoint, r.request_key): r.payload_sha256 for r in _last}
stats = {}

def fetch(endpoint, calls, on_payload=None):
    rows, landed, unchanged, failed = [], 0, 0, []
    for res in client.get_many(calls, MAX_WORKERS):
        now, path, size, digest, outcome = P.utc_now(), None, None, None, "FAILED"
        if res["status_code"] == 200:
            text = API.canonical_json(res["payload"])
            digest = API.sha256(text)
            if last_sha.get((endpoint, res["key"])) == digest:
                outcome, unchanged = "UNCHANGED", unchanged + 1
            else:
                path, size = API.land_response(C.RAW_ROOT, endpoint, res["key"], res["url"], 200, text, now, RUN_ID)
                outcome, landed = "LANDED", landed + 1
                last_sha[(endpoint, res["key"])] = digest
            if on_payload:
                on_payload(res["key"], res["payload"])
        else:
            failed.append(res["key"])
        rows.append((endpoint, res["key"], res["url"], int(res["status_code"]), outcome, res["error"],
                     digest, path, size, now, RUN_ID))
    if rows:
        spark.createDataFrame(rows, spark.table(C.EXTRACT_REGISTER).schema) \
             .write.format("delta").mode("append").saveAsTable(C.EXTRACT_REGISTER)
    stats[endpoint] = {"requests": len(calls), "landed": landed, "unchanged": unchanged, "failed": len(failed)}
    print(f"{endpoint:<10} requests={len(calls):>5}  landed={landed:>5}  unchanged={unchanged:>5}  failed={len(failed)}")
    return failed

## 1. Schedule → games in scope

In [ ]:
with P.log_step(spark, RUN_ID, "bronze", "extract_schedule", WINDOW_LABEL) as step:
    entries = []
    failed = fetch("schedule", [API.schedule_call(a, b, C.GAME_TYPES) for a, b in windows],
                   lambda key, payload: entries.extend(API.games_from_schedule(payload)))
    if failed:
        raise RuntimeError(f"Schedule requests failed: {failed}")
    games = {}
    for g in entries:  # a postponed/suspended game appears twice; the completed entry wins
        if g["game_type"] in C.GAME_TYPES and (g["game_pk"] not in games or g["is_completed"]):
            games[g["game_pk"]] = g
    completed = {pk: g for pk, g in games.items() if g["is_completed"]}
    seasons_seen = sorted({g["season"] for g in games.values() if g["season"]})
    regular_seasons = sorted({g["season"] for g in games.values() if g["game_type"] == "R"})
    venue_ids = {g["venue_id"] for g in games.values() if g["venue_id"]}
    step.records = len(games)
    step.message = f"{len(games)} games in scope, {len(completed)} completed, seasons {seasons_seen}"
    print(step.message)

## 2. Boxscores (one request per completed game)

In [ ]:
with P.log_step(spark, RUN_ID, "bronze", "extract_boxscores", WINDOW_LABEL) as step:
    already = {k for (e, k) in last_sha if e == "boxscore"}
    recheck_from = (yesterday - timedelta(days=LOOKBACK)).isoformat() if MODE == "daily" and not START else None
    targets = sorted(pk for pk, g in completed.items()
                     if str(pk) not in already or (recheck_from and g["official_date"] >= recheck_from))
    seen_players = set()
    failed = fetch("boxscore", [API.boxscore_call(pk) for pk in targets],
                   lambda key, payload: seen_players.update(API.player_ids_from_boxscore(payload)))
    failure_rate = len(failed) / len(targets) if targets else 0
    step.records = len(targets) - len(failed)
    step.message = f"{len(targets)} boxscores requested, {len(failed)} failed"
    if failure_rate > 0.02:
        raise RuntimeError(f"Boxscore failure rate {failure_rate:.1%} above 2%: {failed[:20]}")

## 3. Player profiles (new players only, or all with `refresh_people`)

In [ ]:
def _ids(sql):
    try:
        return {r[0] for r in spark.sql(sql).collect() if r[0] is not None}
    except Exception:  # table not created yet (first run)
        return set()

with P.log_step(spark, RUN_ID, "bronze", "extract_people", WINDOW_LABEL) as step:
    known = _ids(f"SELECT player_id FROM {C.table('silver', 'dim_player')}")
    in_facts = _ids(f"""SELECT player_id FROM {C.table('silver', 'fact_batting_game')}
                        UNION SELECT player_id FROM {C.table('silver', 'fact_pitching_game')}""")
    wanted = (seen_players | in_facts) if REFRESH_PEOPLE else (seen_players | in_facts) - known
    batches = API.chunked(sorted(wanted), 100)
    stamp = P.utc_now().strftime("%Y%m%dT%H%M%S")
    failed = fetch("people", [API.people_call(b, f"{stamp}_{i:03d}") for i, b in enumerate(batches)])
    step.records = len(wanted)
    if failed:
        raise RuntimeError(f"People batches failed: {failed}")

## 4. Teams, venues and official standings

In [ ]:
with P.log_step(spark, RUN_ID, "bronze", "extract_reference", WINDOW_LABEL) as step:
    known_venues = _ids(f"SELECT venue_id FROM {C.table('silver', 'dim_venue')}")
    new_venues = sorted(venue_ids - known_venues)
    failed = fetch("teams", [API.teams_call(s) for s in seasons_seen])
    failed += fetch("venues", [API.venues_call(b, f"{stamp}_{i:03d}") for i, b in enumerate(API.chunked(new_venues, 50))])
    failed += fetch("standings", [API.standings_call(s) for s in regular_seasons])
    step.records = len(seasons_seen) * 2 + len(new_venues)
    if failed:
        raise RuntimeError(f"Reference requests failed: {failed}")

In [ ]:
import pandas as pd
display(pd.DataFrame(stats).T.reset_index(names="endpoint"))